# 1. 데이터 확인

### 데이터 불러오기
> DB파일, csv파일

In [57]:
import pandas as pd
import sqlite3

# DB 파일 경로 설정
db_path = r"C:\Users\user\Desktop\기업 연계 데이터\DB_load\integrated_analysis_통합본.db"

# CSV 파일 경로 설정
csv_path = r"C:\Users\user\Desktop\기업 연계 데이터\data merge\final_merged.csv"

# SQLite 연결
conn = sqlite3.connect(db_path)

# integrated_user_data 테이블 로드
df_db = pd.read_sql("SELECT * FROM movie_master", conn)

# SQLite 연결 종료
conn.close()

# CSV 파일 로드
df_csv = pd.read_csv(csv_path)

# TITLE 고유 개수 확인
db_title_count = df_db["TITLE"].nunique()
csv_title_count = df_csv["TITLE"].nunique()

# 결과 출력
print("df_db의 고유 TITLE 개수:", db_title_count)
print("df_csv의 고유 TITLE 개수:", csv_title_count)


df_db의 고유 TITLE 개수: 14018
df_csv의 고유 TITLE 개수: 5171


### 장르 병합

In [58]:
# TITLE 기준으로 가져올 컬럼만 추출
df_db_mapping = df_db[["TITLE", "GENRE", "COUNTRY", "DURATION"]].copy()

# TITLE 중복 제거
df_db_mapping = df_db_mapping.drop_duplicates(subset="TITLE")

# df_csv에 맞게 컬럼명 변경
df_db_mapping = df_db_mapping.rename(
    columns={
        "GENRE": "genre",
        "COUNTRY": "country",
        "DURATION": "showTM"
    }
)

# TITLE 기준으로 병합
df_csv = df_csv.merge(
    df_db_mapping,
    on="TITLE",
    how="left"
)


# df_csv 총 행 수 확인
print("df_csv 총 행 수:", len(df_csv))

# 추가한 컬럼만 선택
added_cols = ["genre", "country", "showTM"]

# 추가한 컬럼 요약 정보 확인
print("\n추가 컬럼별 결측치 제외 데이터 수")
print(df_csv[added_cols].count())

print("\n추가 컬럼별 결측치 수")
print(df_csv[added_cols].isna().sum())


df_csv 총 행 수: 104619

추가 컬럼별 결측치 제외 데이터 수
genre      102838
country    104587
showTM     104153
dtype: int64

추가 컬럼별 결측치 수
genre      1781
country      32
showTM      466
dtype: int64


In [59]:
# TITLE 기준 분석용 데이터 추출
df_title_level = df_csv[["TITLE", "genre", "country", "showTM"]].dropna(subset=["TITLE"]).copy()

# TITLE별 컬럼 결측 여부 집계
missing_by_title = df_title_level.groupby("TITLE").agg({
    "genre": lambda x: x.isna().all(),
    "country": lambda x: x.isna().all(),
    "showTM": lambda x: x.isna().all()
}).reset_index()

# 고유 TITLE 개수 확인
print("고유 TITLE 개수:", missing_by_title["TITLE"].nunique())

# 각 컬럼별 결측 TITLE 개수 확인
print("genre가 비어 있는 고유 TITLE 개수:", missing_by_title["genre"].sum())
print("country가 비어 있는 고유 TITLE 개수:", missing_by_title["country"].sum())
print("showTM이 비어 있는 고유 TITLE 개수:", missing_by_title["showTM"].sum())


고유 TITLE 개수: 5171
genre가 비어 있는 고유 TITLE 개수: 56
country가 비어 있는 고유 TITLE 개수: 2
showTM이 비어 있는 고유 TITLE 개수: 5


### 시청 기록이 없는 유저 추가

In [60]:
import pandas as pd

# Membership.csv 파일 로드
membership_path = r"C:\Users\user\Desktop\기업 연계 데이터\original data\Membership.csv"
membership = pd.read_csv(membership_path)

# user_no 컬럼명을 uid로 변경
membership = membership.rename(columns={"user_no": "uid"})

# uid 컬럼 자료형 통일
df_csv["uid"] = df_csv["uid"].astype("string")
membership["uid"] = membership["uid"].astype("string")

# df_csv에 존재하지 않는 uid만 membership에서 추출
membership_only = membership[
    membership["uid"].notna() & ~membership["uid"].isin(df_csv["uid"])
].copy()

# df_csv 컬럼 구조에 맞게 정렬 및 없는 컬럼 결측치 처리
membership_only_aligned = membership_only.reindex(columns=df_csv.columns)

# df_csv 아래에 행 추가
df_csv = pd.concat([df_csv, membership_only_aligned], ignore_index=True)

# 결과 확인
print("membership 전체 행 수:", len(membership))
print("df_csv에 없어서 추가된 uid 행 수:", len(membership_only_aligned))
print("행 추가 후 df_csv 총 행 수:", len(df_csv))


membership 전체 행 수: 18183
df_csv에 없어서 추가된 uid 행 수: 3504
행 추가 후 df_csv 총 행 수: 108123


In [61]:
# df_csv의 고유 uid 개수 확인
unique_uid_count = df_csv["uid"].nunique()

print("df_csv의 고유 uid 개수:", unique_uid_count)


df_csv의 고유 uid 개수: 17845


### 원핫인코딩
> 프로모션 여부, 해지방어 여부, 재구매 여부, 본인인증 여부

In [62]:
# Y/N 이진 변환 대상 컬럼 정의
yn_columns = ["promotion_yn", "is_churn_prevented", "repurchase"]

# promotion_yn, is_churn_prevented, repurchase 컬럼 변환
for col in yn_columns:
    df_csv[col] = df_csv[col].map({"Y": 1, "N": 0})

# is_user_verified 컬럼 변환
df_csv["is_user_verified"] = (df_csv["is_user_verified"] == "Y").astype(int)


### payement_device 그룹화

In [ ]:
# payment_device 값 재분류
df_csv["payment_device"] = df_csv["payment_device"].apply(
    lambda x: (
        x if pd.isna(x)
        else x if x in ["ios", "mobile", "pc", "android"]
        else "OTT" if x in ["ott", "ott_cjhello"]
        else "SmartTV"
    )
)

# 결과 확인
print(df_csv["payment_device"].value_counts(dropna=False))


payment_device
android    50700
pc         20770
ios        17899
mobile     16850
SmartTV     1500
OTT          404
Name: count, dtype: int64


# 2. 전처리 